# Pipeline L2C : exploration des données et démonstration sur CLP

Partie 1 : exploration des projets fournis (`data/<PROJET>/`). Partie 2 : pipeline de bout en bout sur CLP, extraction déterministe (sans `--llm`).

In [ ]:
import json
import sys
import tempfile
import time
from collections import Counter
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd
from IPython.display import display

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "l2c_verif").is_dir())
sys.path.insert(0, str(REPO))

DATA = REPO / "data"
PROJECT = DATA / "CLP"
GT = next(PROJECT.glob("*dismatch*.xlsx"), None)
OUT = Path(tempfile.gettempdir()) / "l2c_demo" / "CLP"

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 80)


def page_stats(pdf: Path, min_chars: int = 20) -> tuple[int, int]:
    """(pages, pages with a native text layer)."""
    with fitz.open(pdf) as doc:
        return doc.page_count, sum(1 for page in doc if len(page.get_text("text").strip()) >= min_chars)

## 1. Exploration des données

### 1.1 Projets

Un projet = un PDF de plans `L2C_PLAN_STR_<PROJET>.pdf` et des dessins d'atelier sous `DA/<catégorie>/`.

In [ ]:
projects = sorted(p for p in DATA.iterdir() if p.is_dir())
rows, layer = [], []
for proj in projects:
    plans = sorted(proj.glob("L2C_PLAN_STR_*.pdf"))
    shops = sorted((proj / "DA").rglob("*.pdf"))
    for pdf in shops:
        n, t = page_stats(pdf)
        layer.append({"projet": proj.name, "categorie": pdf.parent.name, "pages": n, "pages_texte": t})
    rows.append({
        "projet": proj.name,
        "pages_plan": sum(page_stats(p)[0] for p in plans),
        "pdf_atelier": len(shops),
        "pages_atelier": sum(r["pages"] for r in layer if r["projet"] == proj.name),
        "verite_terrain": any(proj.glob("*dismatch*.xlsx")),
    })
display(pd.DataFrame(rows))

### 1.2 Couche texte des dessins d'atelier

Pages dont le texte est extractible directement, sur le total, par projet et catégorie. Les autres pages passent par l'OCR Tesseract.

In [ ]:
layer = pd.DataFrame(layer)
agg = layer.groupby(["projet", "categorie"])[["pages_texte", "pages"]].sum()
display((agg["pages_texte"].astype(str) + "/" + agg["pages"].astype(str)).unstack(fill_value="-"))

### 1.3 Index des feuillets du plan CLP

Étape 1 (`l2c_verif.ingest.index_project`) : numéro de feuillet lu dans le cartouche, série, type d'élément déduit de la série, niveau.

In [ ]:
from l2c_verif.ingest import index_project

t0 = time.time()
pages = index_project(PROJECT)
print(f"{len(pages)} pages indexées en {time.time() - t0:.1f} s "
      f"({sum(p.source == 'plan' for p in pages)} plan, {sum(p.source == 'atelier' for p in pages)} atelier)")

sheets = pd.DataFrame([{"page": p.page, "feuillet": p.feuillet, "serie": f"S-{p.series}xx" if p.series else None,
                        "type_element": p.type_element, "niveau": p.niveau}
                       for p in pages if p.source == "plan"])
display(sheets.groupby("serie", dropna=False).agg(
    feuillets=("feuillet", ", ".join),
    type_element=("type_element", lambda s: ", ".join(sorted({t for t in s if t})) or "-")))

### 1.4 Grille d'axes

Étape 2 (`l2c_verif.grid.detect_grid`) : axes reconstruits à partir des bulles (chiffres d'un côté, lettres de l'autre). La grille, commune au plan et à l'atelier, sert de clé de localisation.

In [ ]:
from l2c_verif.extract import implemented_types
from l2c_verif.grid import detect_grid

grid_rows = []
for p in pages:
    if p.type_element not in implemented_types():
        continue
    g = detect_grid(p.lines, p.width, p.height)
    grid_rows.append({"source": p.source, "type_element": p.type_element,
                      "document": p.feuillet if p.source == "plan" else f"{p.categorie} p{p.page}",
                      "axes_verticaux": len(g.verticals), "axes_horizontaux": len(g.horizontals), "grille": g.ok})
grid_df = pd.DataFrame(grid_rows)
display(grid_df.groupby(["source", "type_element"]).agg(
    pages=("grille", "size"), grilles_detectees=("grille", "sum"),
    axes_v_median=("axes_verticaux", "median"), axes_h_median=("axes_horizontaux", "median")))

### 1.5 Texte extrait et coordonnées

Lignes de texte lues par PyMuPDF, en points PDF (1/72 po), origine en haut à gauche. Par type de page : nombre de lignes et part des lignes reconnues comme libellés d'armature par la lecture déterministe (`l2c_verif.rebar.parse_line`). Puis la position de quelques libellés d'un feuillet de colonnes (le texte n'est pas affiché).

In [ ]:
from l2c_verif.rebar import parse_line

text_rows = []
for p in pages:
    if p.type_element not in implemented_types():
        continue
    rebar = sum(1 for ln in p.lines if parse_line(ln["text"]))
    text_rows.append({"source": p.source, "type_element": p.type_element, "lignes": len(p.lines), "libelles_armature": rebar})
text_df = pd.DataFrame(text_rows).groupby(["source", "type_element"]).sum()
text_df["part_armature_%"] = (100 * text_df["libelles_armature"] / text_df["lignes"]).round(1)
display(text_df)

column_sheet = next(p for p in pages if p.source == "plan" and p.type_element == "colonne")
display(pd.DataFrame([{"x0": ln["bbox"][0], "y0": ln["bbox"][1], "x1": ln["bbox"][2], "y1": ln["bbox"][3],
                       "taille": ln["size"], "barres_lues": len(parse_line(ln["text"]))}
                      for ln in column_sheet.lines if parse_line(ln["text"])][:8]))

## 2. Démonstration de bout en bout sur CLP

### 2.1 Exécution

Équivalent CLI : `python -m l2c_verif run --project data/CLP --out results/CLP --gt data/CLP/CLP_dismatch.xlsx`. Ici les sorties vont dans un dossier temporaire pour ne pas modifier `results/`.

In [ ]:
from l2c_verif.pipeline import run_project

t0 = time.time()
summary = run_project(PROJECT, OUT, gt=GT, use_llm=False)
print(f"durée totale : {time.time() - t0:.0f} s")
display(pd.DataFrame(summary["etapes"]))
print("sorties :", sorted(p.name for p in OUT.iterdir() if p.is_file()))

### 2.2 Base JSON (annexe A)

`annexe_a_plan.json` et `annexe_a_atelier.json` : un enregistrement par élément, au schéma de l'annexe A. Nombre d'éléments par type, contrôle des clés, taux de remplissage des attributs d'armature et contrôle des coordonnées.

In [ ]:
from l2c_verif.annex_a import ARMATURE_KEYS, ITEM_KEYS

annex = {src: json.loads((OUT / f"annexe_a_{src}.json").read_text(encoding="utf-8")) for src in ("plan", "atelier")}
display(pd.DataFrame({src: Counter(r["type_element"] for r in recs) for src, recs in annex.items()}).fillna(0).astype(int))

sizes = {}
for p in pages:
    sizes[(p.fichier, p.page)] = (p.width, p.height)
checks = []
for src, recs in annex.items():
    bars = [b for r in recs for b in r["armature"]]
    inside = sum(1 for r in recs if (r["fichier"], r["page"]) in sizes
                 and 0 <= r["x"] <= sizes[(r["fichier"], r["page"])][0] and 0 <= r["y"] <= sizes[(r["fichier"], r["page"])][1])
    checks.append({
        "source": src, "elements": len(recs), "barres": len(bars),
        "cles_conformes": all(list(r) == ITEM_KEYS and all(list(b) == ARMATURE_KEYS for b in r["armature"]) for r in recs),
        "xy_dans_la_page_%": round(100 * inside / len(recs), 1),
        **{f"{k}_%": round(100 * sum(b[k] is not None for b in bars) / max(len(bars), 1), 1)
           for k in ("diametre", "quantite", "espacement_mm", "longueur_mm")},
    })
display(pd.DataFrame(checks))

### 2.3 Constats

Statuts : `conforme`, `non_conforme` (écarts détaillés), `manquant_atelier`, `ajoute_atelier`. Comptes par type d'élément, puis attributs en écart.

In [ ]:
findings = json.loads((OUT / "findings.json").read_text(encoding="utf-8"))
fdf = pd.DataFrame(findings)
display(pd.crosstab(fdf["type_element"], fdf["statut"], margins=True, margins_name="total"))

gaps = pd.DataFrame([{"type_element": f["type_element"], "attribut": e["attribut"]}
                     for f in findings if f["statut"] == "non_conforme" for e in f.get("ecarts", [])])
if not gaps.empty:
    display(pd.crosstab(gaps["type_element"], gaps["attribut"]))

### 2.4 Comptes par feuillet

`summary.json`, repris en tête du rapport PDF. `hors_portee_atelier` : éléments du plan dont le type et le niveau n'ont aucun dessin d'atelier lisible.

In [ ]:
cols = ["feuillet", "type_element", "conforme", "non_conforme", "manquant_atelier", "ajoute_atelier", "hors_portee_atelier"]
sheet_df = pd.DataFrame(summary["par_feuillet"])[cols]
display(sheet_df[sheet_df[cols[2:]].sum(axis=1) > 0])

### 2.5 Évaluation sur la vérité terrain

Rappel : part des écarts connus signalés `non_conforme` au même feuillet et à la même localisation. Précision : part des `non_conforme` qui correspondent à un écart connu.

In [ ]:
ev = summary.get("evaluation") or {}
display(pd.Series({k: ev.get(k) for k in ("rappel", "precision", "n_attendus", "n_trouves", "n_non_conformes", "faux_positifs")}, dtype=object))

### 2.6 Rapport PDF, PDF annoté et manifeste

`report.pdf` : conformités et non-conformités par feuillet, puis liste détaillée des écarts. `annotated.pdf` : écarts encerclés sur le plan et sur le dessin d'atelier, avec lien entre les deux. `run.json` : traçabilité du run.

In [ ]:
for name in ("report.pdf", "annotated.pdf"):
    with fitz.open(OUT / name) as doc:
        print(f"{name} : {doc.page_count} pages, {(OUT / name).stat().st_size / 1e6:.1f} Mo")
run = json.loads((OUT / "run.json").read_text(encoding="utf-8"))
print("mode :", run["mode"], "| durée :", run["duree_s"], "s | sorties :", [k for k, v in run["sorties"].items() if v])